Revenging open ai to extract the relationships between the node entities for each article

In [1]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
ARTICLES_COLLECTION_NAME = os.getenv("MONGO_ARTICLES_NAME")
NODES_COLLECTION_NAME = os.getenv("MONGO_NODES_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, ARTICLES_COLLECTION_NAME, NODES_COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]

    # Define collections
    articles_collection = db[ARTICLES_COLLECTION_NAME]  # Stores articles
    nodes_collection = db[NODES_COLLECTION_NAME]  # Stores extracted nodes

    # Verify connection
    client.admin.command('ping')
    print(f"✅ Connected to MongoDB Atlas! Database: {DB_NAME}")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

#articles_collection.find_one()

✅ Connected to MongoDB Atlas! Database: tuone


In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
openAI_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openAI_api_key)

def ping_openai(client):
    try:
        response = client.models.list()
        print("✅ Successfully connected to OpenAI API!")
        print(f"Available Models: {[model.id for model in response.data]}")
    except Exception as e:
        print(f"❌ OpenAI API Connection Error: {e}")
ping_openai(client)

✅ Successfully connected to OpenAI API!
Available Models: ['gpt-4o-realtime-preview-2024-12-17', 'gpt-4o-audio-preview-2024-12-17', 'dall-e-3', 'o1-2024-12-17', 'dall-e-2', 'gpt-4o-audio-preview-2024-10-01', 'o1-pro-2025-03-19', 'o1-pro', 'o1', 'gpt-4o-realtime-preview-2024-10-01', 'gpt-4o-transcribe', 'gpt-4o-mini-transcribe', 'gpt-4o-realtime-preview', 'babbage-002', 'gpt-4o-mini-tts', 'tts-1-hd-1106', 'text-embedding-3-large', 'gpt-4', 'text-embedding-ada-002', 'omni-moderation-latest', 'computer-use-preview', 'computer-use-preview-2025-03-11', 'tts-1-hd', 'gpt-4o-mini-audio-preview', 'gpt-4o-audio-preview', 'o1-preview-2024-09-12', 'gpt-4o-mini-realtime-preview', 'gpt-4o-mini-realtime-preview-2024-12-17', 'gpt-3.5-turbo-instruct-0914', 'gpt-4o-mini-search-preview', 'tts-1-1106', 'davinci-002', 'gpt-3.5-turbo-1106', 'gpt-4-turbo', 'gpt-4-0125-preview', 'gpt-3.5-turbo-instruct', 'gpt-3.5-turbo', 'gpt-4-turbo-preview', 'chatgpt-4o-latest', 'gpt-4o-mini-search-preview-2025-03-11', 'gpt

In [3]:
import json
from bson import json_util
def fetch_articles(limit=00,offset=0):
    try:
        # Fetch articles with an optional limit
        articles_cursor = (articles_collection.find()
                           .skip(offset)
                           .limit(limit))
        articles = list(articles_cursor)

        if articles:
            print(f"✅ Retrieved {len(articles)} articles from MongoDB.\n")
            for idx, article in enumerate(articles, start=1):
                print(f"--- Article {idx} ---")
                # Convert BSON to JSON using json_util
                article_json = json.dumps(article, indent=4, default=json_util.default)
                print(article_json)
        else:
            print("⚠️ No articles found in the collection.")

        return articles

    except Exception as e:
        print(f"❌ Error fetching articles: {e}")
        return []

# Fetch and print articles in JSON format
articles = fetch_articles(limit=5,offset=15)

✅ Retrieved 5 articles from MongoDB.

--- Article 1 ---
{
    "_id": {
        "$oid": "67b1e4240554959fd725ee05"
    },
    "title": "BASF and TODA reinforce battery production in Japan and USA",
    "paragraphs": [
        {
            "p1": "BASF and TODA strengthen their cooperation as they announce to create the world\u2019s largest calcination facility for high nickel Cathode Active Materials in Japan. They will also combine their manufacturing activities in the States.",
            "p2": "BASF TODA Battery Materials LLC (BTBM), a joint venture between BASF and TODA, has now tripled its high nickel Cathode Active Materials (CAMs) capacity at its Onoda site in Japan.",
            "p3": "Jeffrey Lou, Senior Vice President of Battery Materials at BASF said that with the expansion \u201cBASF (is) further strengthening our market position in high energy cathode active materials for e-mobility applications.\u201d",
            "p4": "A similar move is planned in the States were both

In [4]:
def read_prompt_from_file_only(file_path):
    with open(file_path, 'r') as file:
        prompt = file.read()
    return prompt

def load_function_schema(path):
    with open(path, "r") as f:
        return json.load(f)

def normalize_id(id_str):
    # we need this helper as model is indifferent returning capacity-1 or capacity_1
    return id_str.replace("-", "_") if id_str else id_str

import re

def normalize_type(type_str):
    """
    Normalize a node or entity type to lower_snake_case.
    Examples:
    - "Company" -> "company"
    - "Joint Venture" -> "joint_venture"
    - "  Joint venture  " -> "joint_venture"
    """
    if not type_str:
        return None
    return re.sub(r"\s+", "_", type_str.strip().lower())

def combine_paragraphs(article):
    paragraphs = article.get('paragraphs', [])
    # Handle missing or empty paragraphs
    if not paragraphs:
        print("⚠️ No paragraphs found in the article.")
        return ""

    combined_text = ""
    for para_obj in paragraphs:
        for key in sorted(para_obj.keys()):
            combined_text += para_obj[key].strip() + " "

    return combined_text.strip()

In [5]:
def extract_nodes(article_id, text, PROMPT_PATH, FUNCTION_SCHEMA_PATH):

    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    try:
        completion = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"Here is the article: {text}"}
            ],
            functions = [function_schema],
            function_call = {"name": "extract_clean_tech_entities"},
            temperature=0
        )

        # parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments
        extracted_data = json.loads(function_args)
        print(f"✅ Retrieved nodes {extracted_data}\n")

        if "nodes" not in extracted_data:
            raise ValueError("Expected 'nodes' key in LLM output but not found.")

        nodes_by_type = extracted_data["nodes"]

        # Flatten nodes while retaining their type
        formatted_nodes = []
        for node_type, node_list in nodes_by_type.items():
            for node in node_list:
                formatted_node = {
                    "article_id": article_id,
                    "id": normalize_id(node.get("id")),
                    "type": normalize_type(node.get("type")),
                    "name": node.get("name"),
                    "location": {
                        "city": node.get("location", {}).get("city", ""),
                        "country": node.get("location", {}).get("country", "")
                    } if node.get("location") else None,
                    "amount": node.get("amount"),
                    "status": node.get("status"),
                }
                formatted_nodes.append(formatted_node)

        return formatted_nodes

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error: {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting nodes: {e}")
        return []

def process_articles(limit, offset, PROMPT_PATH, FUNCTION_SCHEMA_PATH, output_path):
    # Retrieve articles with limit & offset, sorted by `_id` for consistency
    articles_to_process = list(
        articles_collection.find(
            {"meta.ID": {"$regex": "^27"}},  # <-- Add this filter
            {"_id": 1, "paragraphs": 1, "validation": 1}       # Keep projection as is
        )
        .sort("_id", -1)  # Sort by MongoDB ObjectId (descending)
        .skip(offset)    # Skip first `offset` articles
        .limit(limit)    # Limit the number of articles
    )

    print(f"🔹 Processing {len(articles_to_process)} articles (Offset: {offset})")

    for article in articles_to_process:
        articleID = str(article["_id"])  # Convert ObjectId to string

        if article.get("validation") is True:
            print(f"⏭️  Skipping Article ID: {articleID} - Article is validated")
            continue

        # Extract text from paragraphs
        text = combine_paragraphs(article)

        # Skip if no text is extracted
        if not text:
            print(f"⚠️ No valid text found for Article ID: {articleID}. Skipping.")
            continue

        print(f"📌 Processing Article ID: {articleID}")

        try:
            # Extract and format nodes
            formatted_nodes = extract_nodes(articleID, text, PROMPT_PATH, FUNCTION_SCHEMA_PATH)

            # Update the article document with extracted nodes and combined text
            update_result = articles_collection.update_one(
                {"_id": article["_id"]},  # Match by article ID
                {"$set": {"combined_text": text, output_path: formatted_nodes or []}}  #return an empty list if no nodes are found
            )

            # Log success
            if update_result.modified_count > 0:
                print(f"✅ Updated Article ID: {articleID} with {len(formatted_nodes)} nodes and combined text.")
            else:
                print(f"⚠️ No updates made for Article ID: {articleID}.")

        except Exception as e:
            print(f"❌ Error processing Article ID {articleID}: {e}")

In [6]:
def extract_events(article_id, text, nodes, PROMPT_PATH, FUNCTION_SCHEMA_PATH, allowed_types=None):

    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    compact_nodes = format_nodes_for_prompt(nodes, allowed_types)
    print(compact_nodes)

    try:
        completion = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"""Here is the article: {text}
                
                Here is the list of known factories:
                {compact_nodes}

                """}
            ],
            functions = [function_schema],
            function_call = {"name": "extract_clean_tech_events"},
            temperature=0
        )

        # parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments
        extracted_data = json.loads(function_args)
        print(f"✅ Retrieved nodes {extracted_data}\n")

        if "events" not in extracted_data:
            raise ValueError("Expected 'events' key in LLM output but not found.")

        events = extracted_data["events"]

        # Flatten nodes while retaining their type
        formatted_events = []
        for event in events:
            formatted_events.append({
                "article_id": article_id,
                "id": normalize_id(event.get("id")),
                "event": event.get("event"),
                "date": event.get("date"),
                "type": event.get("type"),
                "factory_id": event.get("factory_id")
                })

        return formatted_events

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error: {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting nodes: {e}")
        return []

def process_events(limit, offset, PROMPT_PATH, FUNCTION_SCHEMA_PATH, output_path, allowed_types=None):
    # Retrieve articles with limit & offset, sorted by `_id` for consistency
    articles_to_process = list(
        articles_collection.find(
            {"meta.ID": {"$regex": "^27"}},  # <-- filter for electric automobile articles
            {"_id": 1, "paragraphs": 1, "nodes_ben": 1, "validation": 1}       # Keep projection as is
        )
        .sort("_id", -1)  # Sort by MongoDB ObjectId (descending)
        .skip(offset)    # Skip first `offset` articles
        .limit(limit)    # Limit the number of articles
    )

    print(f"🔹 Processing {len(articles_to_process)} articles (Offset: {offset})")

    for article in articles_to_process:
        articleID = str(article["_id"])  # Convert ObjectId to string

        # Check validation status
        if article.get("validation") is True:
            print(f"⏭️  Skipping Article ID: {articleID} - Article is validated")
            continue  

        # Extract text from paragraphs
        text = combine_paragraphs(article)
        nodes = article.get("nodes_ben", [])

        # Skip if no text is extracted
        if not text:
            print(f"⚠️ No valid text found for Article ID: {articleID}. Skipping.")
            continue

        print(f"📌 Processing Article ID: {articleID}")

        try:
            # Extract and format nodes
            formatted_events = extract_events(articleID, text, nodes, PROMPT_PATH, FUNCTION_SCHEMA_PATH, allowed_types)

            # Update the article document with extracted nodes and combined text
            update_result = articles_collection.update_one(
                {"_id": article["_id"]},  # Match by article ID
                {"$set": {"combined_text": text, output_path: formatted_events or []}}  #return an empty list if no events are extracted
            )

            # Log success
            if update_result.modified_count > 0:
                print(f"✅ Updated Article ID: {articleID} with {len(formatted_events)} nodes and combined text.")
            else:
                print(f"⚠️ No updates made for Article ID: {articleID}.")

        except Exception as e:
            print(f"❌ Error processing Article ID {articleID}: {e}")

In [7]:
def format_nodes_for_prompt(nodes, allowed_types=None):
    """
    Format nodes into a clear ID-to-description mapping for GPT prompts.
    Falls back to `amount` for Capacity nodes if `name` is missing.
    """
    lines = ["The following is a list of known entities. You MUST ALWAYS use the ID (left value) when referring to it in a relationship:"]
    print("Allowed types in format_nodes_for_prompt: ", allowed_types)
    for node in nodes:
        node_id = node.get("id")
        node_type = node.get("type")

        name = node.get("name")
        if not name and node_type in {"Capacity", "Investment"}:
            name = node.get("amount")

        if node_id and node_type and name:
            if allowed_types is None or node_type in allowed_types:
                lines.append(f"- ID: {node_id}: Name: {name} ({node_type})")

    return "\n".join(lines)

def extract_relationships(article_id, text, nodes, PROMPT_PATH, FUNCTION_SCHEMA_PATH, allowed_types=None):
    
    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    prompt = read_prompt_from_file_only(PROMPT_PATH)
    
    print(allowed_types)
    compact_nodes = format_nodes_for_prompt(nodes, allowed_types)
    print(compact_nodes)

    try:
        completion = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": prompt},
                {
                "role": "user",
                 "content": f"""Here is the article text: {text}

                Here is the list of known entities:
                {compact_nodes}

                Please extract only the specified relationship types."""
                }
            ],
            functions=[function_schema],
            function_call={"name": "extract_clean_tech_relationships"},
            temperature=0
        )

        # Parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments
        extracted_data = json.loads(function_args)
        print(f"✅ Retrieved relationships {extracted_data}\n")

        if "relationships" not in extracted_data:
            raise ValueError("Expected 'relationships' key in LLM output but not found.")

        formatted_relationships = []
        for rel in extracted_data["relationships"]:
            raw_source = rel.get("source")
            raw_target = rel.get("target")
            norm_source = normalize_id(raw_source)
            norm_target = normalize_id(raw_target)

            print(f"🔍 Normalizing IDs - Source: {raw_source} → {norm_source}, Target: {raw_target} → {norm_target}")

            formatted_relationships.append({
                "article_id": article_id,
                "id": f"{norm_source}_{rel.get('type')}_{norm_target}",
                "source": norm_source,
                "target": norm_target,
                "type": rel.get("type")
            })

        return formatted_relationships

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error (relationships): {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting relationships: {e}")
        return []

In [8]:
def format_events_for_prompt(events):
    lines = ["The following is a list of known events. You MUST ALWAYS use the ID of the event (left value) when referring to it in a relationship:"]
    for event in events:
        event_id = event.get("id")
        event_type = event.get("type")
        event_date = event.get("date")
        if event_id and event_type and event_date:
            lines.append(f"- ID: {event_id}: Event: {event_type} ({event_date})")
    return "\n".join(lines)

def extract_relationships_with_events(article_id, text, nodes, events, PROMPT_PATH, FUNCTION_SCHEMA_PATH, allowed_types=None):
    
    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    prompt = read_prompt_from_file_only(PROMPT_PATH)
    
    compact_nodes = format_nodes_for_prompt(nodes, allowed_types)
    print("Allowed types: ", allowed_types)
    print(compact_nodes)
    compact_events = format_events_for_prompt(events)
    print(compact_events)

    try:
        completion = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": prompt},
                {
                "role": "user",
                 "content": f"""Here is the article text: {text}

                Here is the list of known entities:
                {compact_nodes}

                Here is the list of known events:
                {compact_events}

                Please extract only the specified relationship types."""
                }
            ],
            functions=[function_schema],
            function_call={"name": "extract_clean_tech_relationships"},
            temperature=0
        )

        # Parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments
        extracted_data = json.loads(function_args)
        print(f"✅ Retrieved relationships {extracted_data}\n")

        if "relationships" not in extracted_data:
            raise ValueError("Expected 'relationships' key in LLM output but not found.")

        formatted_relationships = []

        for rel in extracted_data["relationships"]:
            raw_source = rel.get("source")
            raw_target = rel.get("target")
            norm_source = normalize_id(raw_source)
            norm_target = normalize_id(raw_target)

            print(f"🔍 Normalizing IDs - Source: {raw_source} → {norm_source}, Target: {raw_target} → {norm_target}")

            formatted_relationships.append({
                "article_id": article_id,
                "id": f"{norm_source}_{rel.get('type')}_{norm_target}",
                "source": norm_source,
                "target": norm_target,
                "type": rel.get("type")
            })

        return formatted_relationships

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error (relationships): {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting relationships: {e}")
        return []
 
def process_relationships(limit, offset, PROMPT_PATH, FUNCTION_SCHEMA_PATH, relationship_type, allowed_types=None, node_type=None):
    # Query articles that already have nodes_ben and combined_text

    articles_to_process = list(
        articles_collection.find(
            {
                "nodes_ben": {"$exists": True},
                "combined_text": {"$exists": True},
                "meta.ID": {"$regex": "^27"}  # <-- Only this line is added
            },
            {"_id": -1, "combined_text": 1, "nodes_ben": 1, "events_ben": 1, "validation": 1}
        )
        .sort("_id", -1)
        .skip(offset)
        .limit(limit)
    )

    print(f"🔹 Processing {len(articles_to_process)} articles for relationship extraction (Offset: {offset})")

    for article in articles_to_process:
        articleID = str(article["_id"])
        text = article.get("combined_text", "")
        nodes = article.get("nodes_ben", [])
        events = article.get("events_ben", [])

        if article.get("validation") is True:
            print(f"⏭️  Skipping Article ID: {articleID} - Article is validated")
            continue

        if not text or not nodes:
            print(f"⚠️ Missing text or nodes for Article ID: {articleID}. Skipping.")
            continue

        print(f"📌 Processing Article ID: {articleID}")

        try:
            #extract relationships using previously extracted nodes
            if node_type == "entities":
                extracted_relationships = extract_relationships(articleID, text, nodes, PROMPT_PATH, FUNCTION_SCHEMA_PATH, allowed_types)
            elif node_type == "events":
                print("PROCESSING EVENTS nodes")
                extracted_relationships = extract_relationships_with_events(articleID, text, nodes, events, PROMPT_PATH, FUNCTION_SCHEMA_PATH, allowed_types)
            else:
                print("node type is not valid")
                return []
            
            print(f"Debug: Extracted relationships: {extracted_relationships}")

            # Ensure a fallback value is set even if empty
            if extracted_relationships is None:
                extracted_relationships = {"relationships": []}
                print(f"Debug: Extracted relationships is None, setting to empty list: {extracted_relationships}")

            # Update the article document with extracted relationships
            update_result = articles_collection.update_one(
                {"_id": article["_id"]},
                {"$set": {relationship_type: extracted_relationships or []}} #we use an empty list if no relationships are extracted
            )

            if update_result.modified_count > 0:
                print(f"✅ Updated Article ID: {articleID} with {len(extracted_relationships)} relationships.")
            else:
                print(f"⚠️ No update performed for Article ID: {articleID}.")

        except Exception as e:
            print(f"❌ Error processing relationships for Article ID {articleID}: {e}")


In [9]:
n_articles = 50
offset_articles = 0

# named entity recognition (like company, factory, investment)
PROMPT_PATH = "prompts/entities-only.txt"
FUNCTION_SCHEMA_PATH = "schemas/entities.json"

process_articles(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, 'nodes_ben')

# extract events that happen (new-announcemment, construction-start)
PROMPT_PATH = "prompts/events.txt"
FUNCTION_SCHEMA_PATH = "schemas/events.json"

#process_events(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, 'events_ben', allowed_types=["factory"])

# extract ownership relationships
PROMPT_PATH = "prompts/relations-ownership.txt"
FUNCTION_SCHEMA_PATH = "schemas/relations-ownership.json"

process_relationships(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_ownership", allowed_types = ["company", "joint_venture", "factory"], node_type='entities')

# extract financial relationships (origin)
PROMPT_PATH = "prompts/relations-financial-origin.txt"
FUNCTION_SCHEMA_PATH = "schemas/relations-financial-origin.json"

process_relationships(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_financial_origin", allowed_types = ["company", "joint_venture", "investment"], node_type='entities')

# extract financial relationships (technological)
PROMPT_PATH = "prompts/relations-financial-technological.txt"
FUNCTION_SCHEMA_PATH = "schemas/relations-financial-technological.json"

process_relationships(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_financial_technological", allowed_types = ["investment", "capacity", "product", "factory"], node_type='entities')

# extract technological relationships
PROMPT_PATH = "prompts/relations-technological.txt"
FUNCTION_SCHEMA_PATH = "schemas/relations-technological.json"

process_relationships(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_technological", allowed_types = ["capacity", "product", "factory"], node_type='entities')

# extract chronology of investment events
PROMPT_PATH = "prompts/chronology-investment.txt"
FUNCTION_SCHEMA_PATH = "schemas/chronology-investment.json"

#process_relationships(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_chronology_investment", allowed_types=["investment"], node_type = "events")

# extract chronology of capacity events
PROMPT_PATH = "prompts/chronology-capacity.txt"
FUNCTION_SCHEMA_PATH = "schemas/chronology-capacity.json"

#process_relationships(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, "relationships_chronology_capacity", allowed_types=["capacity"], node_type = "events")


🔹 Processing 50 articles (Offset: 0)
⏭️  Skipping Article ID: 67b20c230554959fd7264b95 - Article is validated
⏭️  Skipping Article ID: 67b20c220554959fd7264b94 - Article is validated
⏭️  Skipping Article ID: 67b20c220554959fd7264b93 - Article is validated
⏭️  Skipping Article ID: 67b20c210554959fd7264b92 - Article is validated
📌 Processing Article ID: 67b20c210554959fd7264b91
✅ Retrieved nodes {'nodes': {'companies': [{'id': 'company_tsmc', 'type': 'company', 'name': 'Taiwan Semiconductor Manufacturing Company'}, {'id': 'company_bosch', 'type': 'company', 'name': 'Bosch'}, {'id': 'company_infineon', 'type': 'company', 'name': 'Infineon'}, {'id': 'company_nxp', 'type': 'company', 'name': 'NXP'}], 'joint_ventures': [{'id': 'joint_venture_esmc', 'type': 'joint_venture', 'name': 'European Semiconductor Manufacturing Company'}], 'factories': [{'id': 'factory_dresden', 'type': 'factory', 'name': 'ESMC chip factory', 'location': {'city': 'Dresden', 'country': 'Germany'}}], 'investments': [{'i

In [28]:
articles_collection.find_one({'title': 'Kia EV3 likely will arrive sooner, EV4 delayed'})

{'_id': ObjectId('67b1e82a0554959fd725f5df'),
 'title': 'Kia EV3 likely will arrive sooner, EV4 delayed',
 'paragraphs': [{'p1': 'Kia wants to launch its EV3 in the first half of 2024. The Kia EV4 is to follow at the beginning of 2025, slightly later than previously planned (end of 2024). This was reported by South Korean media, citing a high-ranking Kia manager.',
   'p2': 'The manager in question is Woo-Jeong Woo, Chief Financial Officer and Senior Vice President of Kia. “Electric vehicles are the biggest influence on Kia’s sales and profits. We plan to flexibly respond to market changes in the medium to long term” he said. “The EV3, EV4, and EV5 will be launched sequentially, and we are determined to make these three models successful.”',
   'p3': 'With the EV5 and the following models EV3 and EV4, Kia wants to cover the electric car market at prices between 35,000 and 50,000 US dollars. Kia announced the EV3 and EV4 in the form of studies in October 2023. The former is an electric 

In [16]:
validated_count = articles_collection.count_documents({"validation": True})
print(f"Number of validated articles: {validated_count}")

# If you want to specifically count only electric vehicle articles that are validated:
ev_validated_count = articles_collection.count_documents({
    "validation": True,
    "meta.ID": {"$regex": "^27"}  # Same filter as in your process_relationships function
})
print(f"Number of validated electric vehicle articles: {ev_validated_count}")

Number of validated articles: 41
Number of validated electric vehicle articles: 41
